In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import random
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import gradio as gr
import re # For cleaning strings

# ========== 0. CONFIGURATION & SEED ==========
SEED = 42
EXCLUDE_INDY_500 = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_PATH = './data/f1_model_final_v3.pth' # Changed version for new run

# Column Name Definitions (Centralized)
BASE_CAT_COLS = ['Grand Prix', 'Team', 'Driver', 'Nationality']
ENGINEERED_NUM_COLS = [
    'year', 'Prev_Year_Driver_PTS', 'Prev_Year_Driver_Pos', 'Driver_Experience_Years',
    'Prev_Year_Team_PTS', 'Prev_Year_Team_Pos', 'Team_Experience_Years',
    'Driver_Prev_Season_FL_Count', 'Is_Home_Race'
]
TARGET_COL = 'is_winner'


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

os.makedirs('./data', exist_ok=True)

# ========== 1. HELPER FUNCTIONS ==========
def clean_string(text):
    if isinstance(text, str):
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
    return text

def time_to_seconds(time_str):
    if pd.isna(time_str) or not isinstance(time_str, str) or not time_str:
        return np.nan
    try:
        if re.match(r'^\d+:\d{2}\.\d+$', time_str):
            parts = time_str.split(':')
            if len(parts) == 2:
                m, s_ms = parts; s, ms = s_ms.split('.')
                return int(m) * 60 + int(s) + int(ms) / 1000.0
            elif len(parts) == 3:
                 h, m, s_ms = parts; s, ms = s_ms.split('.')
                 return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000.0
        elif re.match(r'^\d+\.\d+$', time_str):
            s, ms = time_str.split('.')
            return int(s) + int(ms) / 1000.0
    except ValueError: return np.nan
    return np.nan 

GP_COUNTRY_MAP = {
    'British': 'GBR', 'Great Britain': 'GBR', 'Monaco': 'MON', 'Italian': 'ITA', 'Italy': 'ITA',
    'German': 'GER', 'Germany': 'GER', 'Belgian': 'BEL', 'Belgium': 'BEL', 'French': 'FRA', 'France': 'FRA',
    'Dutch': 'NED', 'Spanish': 'ESP', 'Spain': 'ESP', 'Brazilian': 'BRA', 'Brazil': 'BRA',
    'Japanese': 'JPN', 'Japan': 'JPN', 'Canadian': 'CAN', 'Canada': 'CAN', 'Austrian': 'AUT', 'Austria': 'AUT',
    'Hungarian': 'HUN', 'Hungary': 'HUN', 'Mexican': 'MEX', 'Mexico': 'MEX', 'Australian': 'AUS', 'Australia': 'AUS',
    'United States': 'USA', 'USA': 'USA', 'Swiss': 'SUI', 'Switzerland': 'SUI',
}

def get_country_from_gp(gp_name):
    if not isinstance(gp_name, str): return None
    for key, country_code in GP_COUNTRY_MAP.items():
        if key in gp_name: return country_code
    return None

# ========== 2. DATA LOADING AND INITIAL CLEANING ==========
def load_and_clean_data(exclude_indy=True):
    print("Loading and cleaning data...")
    df_map = {}
    try:
        df_map['winners'] = pd.read_csv('./data/winners.csv', encoding='utf-8')
        df_map['drivers'] = pd.read_csv('./data/drivers_updated.csv', encoding='utf-8')
        df_map['teams'] = pd.read_csv('./data/teams_updated.csv', encoding='utf-8')
        df_map['fastest_laps'] = pd.read_csv('./data/fastest_laps_updated.csv', encoding='utf-8')
    except FileNotFoundError as e:
        print(f"Error: Missing CSV file. {e}")
        return {key: None for key in ['winners', 'drivers', 'teams', 'fastest_laps']}

    string_cols_map = {
        'winners': ['Grand Prix', 'Winner', 'Car'],
        'drivers': ['Driver', 'Nationality', 'Car'],
        'teams': ['Team'],
        'fastest_laps': ['Grand Prix', 'Driver', 'Car']
    }
    for name, df in df_map.items():
        if df is None: continue
        for col in string_cols_map.get(name, []):
            if col in df.columns: df[col] = df[col].apply(clean_string)

    if df_map['winners'] is not None and 'Date' in df_map['winners'].columns:
        df_map['winners']['year'] = pd.to_datetime(df_map['winners']['Date'], errors='coerce').dt.year
        df_map['winners'].dropna(subset=['year'], inplace=True)
        df_map['winners']['year'] = df_map['winners']['year'].astype(int)
    elif df_map['winners'] is not None:
        raise ValueError("winners.csv is missing 'Date' column.")

    for name, df_name_str in [('drivers', 'drivers_updated.csv'), ('teams', 'teams_updated.csv'), ('fastest_laps', 'fastest_laps_updated.csv')]:
        df = df_map.get(name)
        if df is None: continue
        if 'year' in df.columns:
            df['year'] = pd.to_numeric(df['year'], errors='coerce')
            df.dropna(subset=['year'], inplace=True)
            df['year'] = df['year'].astype(int)
        else: raise ValueError(f"{df_name_str} is missing 'year' column.")
        if 'Pos' in df.columns: df['Pos'] = pd.to_numeric(df['Pos'], errors='coerce')
            
    if df_map['drivers'] is not None:
        if 'Car' in df_map['drivers'].columns:
            df_map['drivers'].rename(columns={'Car': 'Team'}, inplace=True)
        else: raise ValueError("drivers_updated.csv is missing 'Car' column (expected as 'Team').")

    if exclude_indy:
        print("Excluding Indianapolis 500 races...")
        for name in ['winners', 'fastest_laps']:
            df = df_map.get(name)
            if df is not None and 'Grand Prix' in df.columns:
                 df_map[name] = df[~df['Grand Prix'].str.contains("Indianapolis 500", na=False)]
    
    print("Data loading and initial cleaning complete.")
    return df_map['winners'], df_map['drivers'], df_map['teams'], df_map['fastest_laps']

# ========== 3. FEATURE ENGINEERING ==========
def _calculate_lagged_features(df, entity_col, val_col, new_col_name_pts, pos_col, new_col_name_pos, default_pos_val):
    if df is None or df.empty: return df
    df = df.sort_values(by=[entity_col, 'year'])
    if val_col in df.columns:
        df[new_col_name_pts] = df.groupby(entity_col)[val_col].shift(1).fillna(0)
    else: df[new_col_name_pts] = 0
    if pos_col in df.columns:
        df[new_col_name_pos] = df.groupby(entity_col)[pos_col].shift(1).fillna(default_pos_val)
    else: df[new_col_name_pos] = default_pos_val
    return df

def _calculate_experience(df, entity_col, new_col_name):
    if df is None or df.empty or 'year' not in df.columns or entity_col not in df.columns:
        if df is not None: df[new_col_name] = 0
        return df
    first_year_map = df.groupby(entity_col)['year'].min().rename(f"{entity_col}_first_year")
    df = df.merge(first_year_map, on=entity_col, how='left')
    if f"{entity_col}_first_year" in df.columns:
        df[new_col_name] = df['year'] - df[f"{entity_col}_first_year"]
        df.drop(columns=[f"{entity_col}_first_year"], inplace=True)
    else: df[new_col_name] = 0
    return df

def engineer_features(drivers_df, teams_df, fastest_laps_df, default_prev_pos_driver, default_prev_pos_team):
    print("Engineering features...")
    if drivers_df is None: 
        print("Error: drivers_df is None in engineer_features. Cannot proceed.")
        return None

    # Driver Features
    drivers_df = _calculate_lagged_features(drivers_df, 'Driver', 'PTS', 'Prev_Year_Driver_PTS', 'Pos', 'Prev_Year_Driver_Pos', default_prev_pos_driver)
    drivers_df = _calculate_experience(drivers_df, 'Driver', 'Driver_Experience_Years')

    # Team Features (calculated on teams_df then merged to drivers_df)
    if teams_df is not None and not teams_df.empty:
        teams_df = _calculate_lagged_features(teams_df, 'Team', 'PTS', 'Prev_Year_Team_PTS', 'Pos', 'Prev_Year_Team_Pos', default_prev_pos_team)
        teams_df = _calculate_experience(teams_df, 'Team', 'Team_Experience_Years')
        
        drivers_df = drivers_df.merge(
            teams_df[['Team', 'year', 'Prev_Year_Team_PTS', 'Prev_Year_Team_Pos', 'Team_Experience_Years']],
            on=['Team', 'year'], how='left'
        )
        for col in ['Prev_Year_Team_PTS', 'Team_Experience_Years']: drivers_df[col] = drivers_df[col].fillna(0)
        drivers_df['Prev_Year_Team_Pos'] = drivers_df['Prev_Year_Team_Pos'].fillna(default_prev_pos_team)
    else: # If teams_df is not available, add placeholder columns to drivers_df
        drivers_df['Prev_Year_Team_PTS'] = 0
        drivers_df['Prev_Year_Team_Pos'] = default_prev_pos_team
        drivers_df['Team_Experience_Years'] = 0

    # Fastest Lap Features
    if fastest_laps_df is not None and not fastest_laps_df.empty and 'Driver' in fastest_laps_df.columns and 'year' in fastest_laps_df.columns:
        driver_fl_yearly_counts = fastest_laps_df.groupby(['year', 'Driver']).size().reset_index(name='FL_Count_In_Year')
        driver_fl_yearly_counts = driver_fl_yearly_counts.sort_values(by=['Driver', 'year'])
        driver_fl_yearly_counts['Driver_Prev_Season_FL_Count'] = driver_fl_yearly_counts.groupby('Driver')['FL_Count_In_Year'].shift(1).fillna(0)
        drivers_df = drivers_df.merge(
            driver_fl_yearly_counts[['Driver', 'year', 'Driver_Prev_Season_FL_Count']],
            on=['Driver', 'year'], how='left'
        )
        drivers_df['Driver_Prev_Season_FL_Count'] = drivers_df['Driver_Prev_Season_FL_Count'].fillna(0)
    else:
        drivers_df['Driver_Prev_Season_FL_Count'] = 0
        
    print("Feature engineering complete.")
    return drivers_df

# ========== 4. DATA RESTRUCTURING FOR MODELING ==========
def restructure_for_modeling(winners_df, drivers_df_with_features, default_prev_pos_driver_val, default_prev_pos_team_val):
    print("Restructuring data for modeling...")
    if winners_df is None or drivers_df_with_features is None or winners_df.empty:
        return pd.DataFrame()

    all_samples = []
    if drivers_df_with_features.empty or 'year' not in drivers_df_with_features.columns:
        return pd.DataFrame()
        
    drivers_grouped_by_year = {year: group for year, group in drivers_df_with_features.groupby('year')}

    for _, race in winners_df.iterrows():
        current_year, current_gp_name, actual_winner_name = race['year'], race['Grand Prix'], race['Winner']
        race_country = get_country_from_gp(current_gp_name)

        if current_year not in drivers_grouped_by_year: continue
        year_drivers_df = drivers_grouped_by_year[current_year]

        for _, p in year_drivers_df.iterrows(): # participant abbreviated as p
            p_nat = p.get('Nationality'); p_driver = p.get('Driver', 'Unknown')
            is_home = 1 if race_country and isinstance(p_nat, str) and race_country == p_nat else 0
            p_driver_str = str(p_driver) if not isinstance(p_driver, str) else p_driver
            
            all_samples.append({
                'year': current_year, 'Grand Prix': current_gp_name, 'Driver': p_driver_str,
                'Team': p.get('Team', 'Unknown Team'), 
                'Nationality': p_nat if isinstance(p_nat, str) else 'Unknown',
                'Prev_Year_Driver_PTS': p.get('Prev_Year_Driver_PTS', 0),
                'Prev_Year_Driver_Pos': p.get('Prev_Year_Driver_Pos', default_prev_pos_driver_val),
                'Driver_Experience_Years': p.get('Driver_Experience_Years', 0),
                'Prev_Year_Team_PTS': p.get('Prev_Year_Team_PTS', 0),
                'Prev_Year_Team_Pos': p.get('Prev_Year_Team_Pos', default_prev_pos_team_val),
                'Team_Experience_Years': p.get('Team_Experience_Years', 0),
                'Driver_Prev_Season_FL_Count': p.get('Driver_Prev_Season_FL_Count', 0),
                'Is_Home_Race': is_home,
                'is_winner': 1 if p_driver_str == actual_winner_name else 0
            })
    
    if not all_samples: return pd.DataFrame()
    print("Data restructuring for modeling complete.")
    return pd.DataFrame(all_samples)

# ========== 5. PREPROCESSING (ENCODING & SCALING) ==========
def preprocess_data(train_df, test_df, cat_cols, num_cols):
    print("Preprocessing data (encoding and scaling)...")
    label_encoders = {}; cat_feat_dims = {}
    active_cat_cols = [c for c in cat_cols if c in train_df.columns]

    for col in active_cat_cols:
        le = LabelEncoder()
        train_df[col] = train_df[col].astype(str)
        test_df[col] = test_df[col].astype(str) if col in test_df else pd.Series(['Unknown'] * len(test_df), name=col, index=test_df.index)
        if col not in test_df.columns: test_df[col] = 'Unknown' # Ensure column exists

        le.fit(train_df[col])
        train_df[col] = le.transform(train_df[col])
        known_classes = set(le.classes_)
        unknown_idx = len(le.classes_)
        test_df[col] = test_df[col].apply(lambda x: le.transform([x])[0] if x in known_classes else unknown_idx)
        label_encoders[col] = le
        cat_feat_dims[col] = unknown_idx + 1

    scaler = StandardScaler()
    active_num_cols = [n for n in num_cols if n in train_df.columns]
    if active_num_cols:
        train_df[active_num_cols] = scaler.fit_transform(train_df[active_num_cols])
        # Ensure test_df has all active_num_cols before transforming
        for ncol in active_num_cols:
            if ncol not in test_df.columns: 
                print(f"Warning: Numerical column '{ncol}' not found in test_df. Filling with 0.")
                test_df[ncol] = 0
        test_df[active_num_cols] = scaler.transform(test_df[active_num_cols])
    
    print("Preprocessing complete.")
    return train_df, test_df, label_encoders, scaler, cat_feat_dims, active_cat_cols, active_num_cols

# ========== 6. DATASET & DATALOADER (No major changes needed from your version) ==========
class F1Dataset(Dataset):
    def __init__(self, df_data, cat_cols_ordered, num_cols_ordered, target_col):
        self.actual_cat_cols = [c for c in cat_cols_ordered if c in df_data.columns]
        self.actual_num_cols = [n for n in num_cols_ordered if n in df_data.columns]

        self.cat = df_data[self.actual_cat_cols].values if self.actual_cat_cols else np.empty((len(df_data), 0), dtype=np.long) # Ensure long for indices
        self.num = df_data[self.actual_num_cols].values if self.actual_num_cols else np.empty((len(df_data), 0), dtype=np.float32)
        
        self.y = df_data[target_col].values
        self.num_categorical_feats = self.cat.shape[1]
        self.num_numerical_feats = self.num.shape[1]

    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        cat_f = self.cat[idx] if self.num_categorical_feats > 0 else np.array([], dtype=np.long)
        num_f = self.num[idx] if self.num_numerical_feats > 0 else np.array([], dtype=np.float32)
        return torch.tensor(cat_f, dtype=torch.long), torch.tensor(num_f, dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)

# ========== 7. MODEL DEFINITION (No major changes needed from your version) ==========
class F1DNN(nn.Module):
    def __init__(self, cat_dims_ordered_list, num_numerical_features, emb_dim=32, hidden_dim=128, dropout_rate=0.4):
        super().__init__()
        self.embeddings = nn.ModuleList()
        total_emb_output_dim = 0
        if cat_dims_ordered_list:
            for num_unique_values in cat_dims_ordered_list:
                self.embeddings.append(nn.Embedding(num_unique_values, emb_dim))
                total_emb_output_dim += emb_dim
        
        self.num_numerical_features = num_numerical_features
        if self.num_numerical_features > 0: self.bn_num = nn.BatchNorm1d(num_numerical_features)
        
        self.fc1_input_dim = total_emb_output_dim + num_numerical_features
        if self.fc1_input_dim == 0: raise ValueError("Model has no input features for fc1.")

        self.fc1 = nn.Linear(self.fc1_input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc3 = nn.Linear(hidden_dim // 2, 1) 

    def forward(self, x_cat, x_num):
        current_features = []
        if self.embeddings:
            if x_cat.shape[1] != len(self.embeddings): raise ValueError(f"x_cat has {x_cat.shape[1]} features, model has {len(self.embeddings)} embeddings.")
            x_emb_list = [emb_layer(x_cat[:, i]) for i, emb_layer in enumerate(self.embeddings)]
            current_features.append(torch.cat(x_emb_list, dim=1))
        
        if self.num_numerical_features > 0:
            if x_num.shape[1] != self.num_numerical_features: raise ValueError(f"x_num has {x_num.shape[1]} features, model expects {self.num_numerical_features}.")
            if x_num.shape[1] > 0: current_features.append(self.bn_num(x_num))
        
        if not current_features: raise ValueError("Model received no input features to concatenate.")
        x = torch.cat(current_features, dim=1)
        
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x))) # Added dropout after fc2 as well
        x = self.fc3(x)
        return x

# ========== 8. TRAINING FUNCTION (No major changes needed from your version) ==========
def train_model(model, trainloader, testloader, n_epoch=30, lr=5e-4, patience=7, model_path=MODEL_PATH): # Use global MODEL_PATH
    print(f"Training on {DEVICE}...")
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    best_test_loss, no_improve_epochs = np.inf, 0
    train_losses, test_losses = [], []

    for epoch in range(n_epoch):
        model.train(); epoch_train_loss = 0
        if not trainloader: print("Warning: Trainloader is empty."); break
        for x_cat, x_num, y_true in trainloader:
            x_cat, x_num, y_true = x_cat.to(DEVICE), x_num.to(DEVICE), y_true.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x_cat, x_num)
            loss = criterion(logits, y_true.unsqueeze(1))
            loss.backward(); optimizer.step()
            epoch_train_loss += loss.item()
        avg_train_loss = epoch_train_loss / len(trainloader) if trainloader else 0.0
        train_losses.append(avg_train_loss)

        model.eval(); epoch_test_loss = 0
        current_eval_loss = avg_train_loss # Default if no testloader
        if testloader and len(testloader.dataset) > 0: # Check if testloader has data
            with torch.no_grad():
                for x_cat, x_num, y_true in testloader:
                    x_cat, x_num, y_true = x_cat.to(DEVICE), x_num.to(DEVICE), y_true.to(DEVICE)
                    logits = model(x_cat, x_num)
                    loss = criterion(logits, y_true.unsqueeze(1))
                    epoch_test_loss += loss.item()
            avg_test_loss = epoch_test_loss / len(testloader) if testloader else float('inf')
            test_losses.append(avg_test_loss); current_eval_loss = avg_test_loss
        else: test_losses.append(avg_train_loss) # Log train loss if no test loss

        print(f"Epoch {epoch+1}/{n_epoch} | Train Loss: {avg_train_loss:.4f} | Test Loss: {test_losses[-1]:.4f}")
        if current_eval_loss < best_test_loss:
            best_test_loss = current_eval_loss; torch.save(model.state_dict(), model_path); no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            if no_improve_epochs >= patience: print(f"Early stopping after {patience} epochs."); break
    
    if train_losses and test_losses:
        plt.figure(figsize=(10,6)); plt.plot(train_losses,label='Train'); plt.plot(test_losses,label='Test')
        plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss Curve'); plt.legend(); plt.grid(True)
        plt.savefig('./data/loss_curve_final.png'); plt.close(); print("Loss curve saved.")
    return train_losses, test_losses

# ========== 9. EVALUATION FUNCTION (No major changes needed from your version) ==========
def evaluate_model(model, dataloader, model_path_to_load=MODEL_PATH): # Use global MODEL_PATH
    if dataloader is None or not hasattr(dataloader, 'dataset') or len(dataloader.dataset) == 0:
        print("Evaluation dataloader empty. Skipping."); return
    if model_path_to_load and os.path.exists(model_path_to_load):
        try: model.load_state_dict(torch.load(model_path_to_load, map_location=DEVICE)); print(f"Loaded: {model_path_to_load}")
        except Exception as e: print(f"Error loading {model_path_to_load}: {e}.")
    elif model_path_to_load: print(f"Warning: {model_path_to_load} not found.")

    model.eval(); all_y_true, all_y_pred_classes = [], []
    with torch.no_grad():
        for x_cat, x_num, y_true in dataloader:
            x_cat,x_num,y_true=x_cat.to(DEVICE),x_num.to(DEVICE),y_true.to(DEVICE)
            logits = model(x_cat, x_num); probs = torch.sigmoid(logits)
            preds = (probs > 0.5).squeeze().int()
            all_y_true.extend(y_true.cpu().numpy())
            all_y_pred_classes.extend(preds.cpu().numpy() if preds.ndim > 0 else [preds.item()])
    if not all_y_true: print("No data evaluated."); return
    print(f"\nAccuracy: {accuracy_score(all_y_true, all_y_pred_classes):.4f}")
    print(classification_report(all_y_true, all_y_pred_classes, target_names=['Not Winner(0)','Winner(1)'], zero_division=0))
    print(f"Confusion Matrix:\n{confusion_matrix(all_y_true, all_y_pred_classes)}")
    
# ========== 10. GRADIO SETUP & PREDICTION LOGIC ==========
# Globals for Gradio, to be populated in main
GRADIO_VARS = {
    "label_encoders": {}, "scaler": None, "model": None, 
    "driver_info_by_year": {}, "all_years": [], "all_gps": [],
    "cat_cols_ordered": [], "num_cols_ordered": [],
    "default_prev_pos_driver": 50, "default_prev_pos_team": 20
}

def gradio_predict_winner_probabilities(year_input, grand_prix_input):
    gv = GRADIO_VARS # Shorthand
    scaler_needed = bool(gv["num_cols_ordered"])
    if gv["model"] is None or not gv["label_encoders"] or (scaler_needed and gv["scaler"] is None):
        return "Error: Model or preprocessors not loaded."
    try: year = int(year_input)
    except ValueError: return "Error: Invalid year."
    if not grand_prix_input: return "Error: Grand Prix input empty."
    if year not in gv["driver_info_by_year"] or not gv["driver_info_by_year"][year]:
        return f"No driver data for year {year}."

    participants = gv["driver_info_by_year"][year]; all_results = []
    gv["model"].eval()

    for p_info in participants:
        cat_vals = []; num_vals = []
        if gv["cat_cols_ordered"]:
            for col_name in gv["cat_cols_ordered"]:
                le = gv["label_encoders"][col_name]; val_to_encode = None
                if col_name == 'Grand Prix': val_to_encode = grand_prix_input
                elif col_name == 'Driver': val_to_encode = p_info.get('Driver', 'Unknown')
                elif col_name == 'Team': val_to_encode = p_info.get('Team', 'Unknown')
                elif col_name == 'Nationality': val_to_encode = p_info.get('Nationality', 'Unknown')
                val_to_encode = str(val_to_encode)
                cat_vals.append(le.transform([val_to_encode])[0] if val_to_encode in set(le.classes_) else len(le.classes_))
            x_cat_tensor = torch.tensor([cat_vals], dtype=torch.long).to(DEVICE)
        else: x_cat_tensor = torch.empty(1, 0, dtype=torch.long).to(DEVICE)
        
        if gv["num_cols_ordered"]:
            for col_name in gv["num_cols_ordered"]:
                val = 0.0 # Default for any numeric feature
                if col_name == 'year': val = float(year)
                elif col_name == 'Is_Home_Race':
                    rc_g = get_country_from_gp(grand_prix_input); p_n_g = p_info.get('Nationality')
                    val = 1.0 if rc_g and isinstance(p_n_g, str) and rc_g == p_n_g else 0.0
                else: # For other pre-calculated lagged features
                    val = p_info.get(col_name, 0 if 'PTS' in col_name or 'Count' in col_name or 'Exp' in col_name 
                                     else (gv["default_prev_pos_driver"] if 'Driver_Pos' in col_name else gv["default_prev_pos_team"])
                                     )
                num_vals.append(float(val))
            x_num_df = pd.DataFrame([num_vals], columns=gv["num_cols_ordered"])
            x_num_tensor = torch.tensor(gv["scaler"].transform(x_num_df), dtype=torch.float32).to(DEVICE)
        else: x_num_tensor = torch.empty(1, 0, dtype=torch.float32).to(DEVICE)

        with torch.no_grad():
            logits = gv["model"](x_cat_tensor, x_num_tensor)
            probability = torch.sigmoid(logits).cpu().item()
        all_results.append((p_info.get('Driver','N/A'), p_info.get('Team','N/A'), probability))

    all_results.sort(key=lambda x: x[2], reverse=True)
    output_text = f"Predictions for {grand_prix_input}, {year}:\n" + \
                  "\n".join([f"{i}. {d} ({t}): {p:.2%}" for i,(d,t,p) in enumerate(all_results[:10],1)])
    return output_text

# ========== 11. MAIN EXECUTION ==========
if __name__ == '__main__':
    gv = GRADIO_VARS # Use the global dict for Gradio vars

    winners_df, drivers_raw_df, teams_raw_df, fastest_laps_raw_df = load_and_clean_data(EXCLUDE_INDY_500)
    if any(df is None for df in [winners_df, drivers_raw_df]): # drivers_raw_df is essential
        print("Exiting due to critical data loading errors."); exit()

    # Calculate default previous positions based on loaded raw data
    def_prev_pos_drv = 50; def_prev_pos_team = 20 # Fallback defaults
    if drivers_raw_df is not None and 'Pos' in drivers_raw_df.columns and not drivers_raw_df['Pos'].dropna().empty:
        max_p_d = drivers_raw_df['Pos'].max(skipna=True)
        def_prev_pos_drv = (int(max_p_d)+5) if pd.notna(max_p_d) and max_p_d > 0 else 50
    if teams_raw_df is not None and 'Pos' in teams_raw_df.columns and not teams_raw_df['Pos'].dropna().empty:
        max_p_t = teams_raw_df['Pos'].max(skipna=True)
        def_prev_pos_team = (int(max_p_t)+5) if pd.notna(max_p_t) and max_p_t > 0 else 20
    gv["default_prev_pos_driver"] = def_prev_pos_drv
    gv["default_prev_pos_team"] = def_prev_pos_team

    drivers_featured_df = engineer_features(drivers_raw_df, teams_raw_df, fastest_laps_raw_df, def_prev_pos_drv, def_prev_pos_team)
    if drivers_featured_df is None or drivers_featured_df.empty: print("Exiting: drivers_featured_df empty."); exit()

    modeling_df = restructure_for_modeling(winners_df, drivers_featured_df, def_prev_pos_drv, def_prev_pos_team)
    if modeling_df is None or modeling_df.empty: print("Exiting: modeling_df empty."); exit()
    
    modeling_df['race_id'] = modeling_df['year'].astype(str) + "_" + modeling_df['Grand Prix'].astype(str)
    unique_race_ids = modeling_df['race_id'].unique()

    if len(unique_race_ids) < 2:
        print("Warning: Not enough unique races for split. Using all data for train/test.")
        train_df, test_df = modeling_df.copy(), modeling_df.copy()
    else:
        train_ids, test_ids = train_test_split(unique_race_ids, test_size=0.2, random_state=SEED, shuffle=True)
        train_df = modeling_df[modeling_df['race_id'].isin(train_ids)].copy()
        test_df = modeling_df[modeling_df['race_id'].isin(test_ids)].copy()
    
    if train_df.empty: print("Error: Train DF empty after split."); exit()
    for df_ in [train_df, test_df]:
        if not df_.empty and 'race_id' in df_.columns: df_.drop(columns=['race_id'], inplace=True)

    # NaN Imputation
    print("\nNaN check and imputation:")
    active_cols = BASE_CAT_COLS + ENGINEERED_NUM_COLS # Use defined lists
    for col in active_cols:
        if col in train_df.columns and train_df[col].isnull().any():
            print(f"NaNs in train_df['{col}']: {train_df[col].isnull().sum()}. Imputing...")
            fill_value = 'Unknown' if train_df[col].dtype == 'object' or pd.api.types.is_string_dtype(train_df[col]) \
                                   else train_df[col].median()
            train_df[col] = train_df[col].fillna(fill_value)
            if col in test_df.columns: test_df[col] = test_df[col].fillna(fill_value)
        # Ensure all defined columns exist, if not, add them with default (important for consistency)
        for df_set in [train_df, test_df]:
            if col not in df_set.columns:
                 df_set[col] = 'Unknown' if col in BASE_CAT_COLS else 0


    train_df, test_df, gv["label_encoders"], gv["scaler"], cat_feat_dims_map, \
        gv["cat_cols_ordered"], gv["num_cols_ordered"] = preprocess_data(train_df, test_df, BASE_CAT_COLS, ENGINEERED_NUM_COLS)

    train_ds = F1Dataset(train_df, gv["cat_cols_ordered"], gv["num_cols_ordered"], TARGET_COL)
    test_ds = F1Dataset(test_df, gv["cat_cols_ordered"], gv["num_cols_ordered"], TARGET_COL) if not test_df.empty else None
    if not train_ds: print("Error: Training dataset empty."); exit()

    weights = [1./(train_df[TARGET_COL].value_counts().get(t,1)+1e-6) for t in train_df[TARGET_COL]]
    sampler = WeightedRandomSampler(torch.DoubleTensor(weights), len(weights))
    train_loader = DataLoader(train_ds, batch_size=256, sampler=sampler)
    test_loader = DataLoader(test_ds, batch_size=256, shuffle=False) if test_ds and len(test_ds)>0 else None

    ordered_cat_dims = [cat_feat_dims_map[col] for col in gv["cat_cols_ordered"] if col in cat_feat_dims_map]
    num_num_feats = len(gv["num_cols_ordered"])
    gv["model"] = F1DNN(ordered_cat_dims, num_num_feats).to(DEVICE)
    
    TRAIN_FLAG = not os.path.exists(MODEL_PATH)
    if TRAIN_FLAG:
        print(f"Training new model: {MODEL_PATH} not found or TRAIN_FLAG is True.")
        eff_test_loader = test_loader if test_loader and len(test_loader.dataset) > 0 else train_loader # Use train if test is bad
        if not train_loader or len(train_loader.dataset) == 0 : print("Error: Train loader empty. Skipping.")
        else: train_model(gv["model"], train_loader, eff_test_loader, n_epoch=50, patience=10)
    else: print(f"Skipping training. Loading from {MODEL_PATH}")

    print("\n--- Final Evaluation ---")
    eff_eval_loader = test_loader if test_loader and len(test_loader.dataset) > 0 else train_loader
    if not eff_eval_loader or len(eff_eval_loader.dataset) == 0: print("No data for final eval.")
    else: evaluate_model(gv["model"], eff_eval_loader, model_path_to_load=MODEL_PATH if not TRAIN_FLAG else None)
    
    # Populate Gradio dropdown choices
    if drivers_featured_df is not None and not drivers_featured_df.empty:
        temp_df_gradio = drivers_featured_df.copy() # Use df with all original + engineered features
        for yr, grp in temp_df_gradio.groupby('year'): gv["driver_info_by_year"][yr] = grp.to_dict('records')
        if 'year' in drivers_featured_df.columns: gv["all_years"] = sorted(list(drivers_featured_df['year'].unique()))
    if not gv["all_years"]: gv["all_years"] = [2023] # Fallback
    if winners_df is not None and not winners_df.empty and 'Grand Prix' in winners_df.columns:
         gv["all_gps"] = sorted(list(winners_df['Grand Prix'].unique()))
    if not gv["all_gps"]: gv["all_gps"] = ["Monaco Grand Prix"] # Fallback
    
    print("\nLaunching Gradio Interface...")
    with gr.Blocks(theme=gr.themes.Soft()) as demo:
        gr.Markdown("# F1 Grand Prix Winner Probability Predictor")
        gr.Markdown("Predicts the win probability for each participating driver in a selected F1 race.")
        with gr.Row():
            year_dd = gr.Dropdown(label="Year", choices=gv["all_years"], value=gv["all_years"][-1] if gv["all_years"] else None)
            gp_dd = gr.Dropdown(label="Grand Prix", choices=gv["all_gps"], value=gv["all_gps"][0] if gv["all_gps"] else None)
        predict_btn = gr.Button("Predict Probabilities", variant="primary")
        output_tb = gr.Textbox(label="Predicted Win Probabilities (Top 10)", lines=12, interactive=False)
        predict_btn.click(gradio_predict_winner_probabilities, inputs=[year_dd, gp_dd], outputs=[output_tb])
        with gr.Accordion("Training Information", open=False):
            gr.Image(value="./data/loss_curve_final.png", label="Loss Curve", show_label=False, visible=os.path.exists("./data/loss_curve_final.png"))
            gr.Markdown(value="Loss curve image not found." if not os.path.exists("./data/loss_curve_final.png") else "")
    demo.launch(share=False)
